In [46]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing import image as keras_image
import import_ipynb
from MLPClass import Layer, MLP, Conv

In [47]:
class DataLoader:
    def __init__(self, target_image_size=(64,64)):
        self.target_image_size = target_image_size
        self.vectorizer = TfidfVectorizer()

    def load(self,source,labels=None,classes=None,label_col=None):
        if isinstance(source, str) and os.path.isfile(source) and \
           source.lower().endswith(('.jpg','.jpeg','.png')):
            raise ValueError(
                "Sorry. Fit() requires a fuller dataset, not a single image file.\n"
                "Please pass a folder path with class sub-folders for image training."
            )
        if isinstance(source,str) and source.lower().endswith('.csv'):
            return self._load_csv(source,label_col=label_col)
        if isinstance(source,str) and os.path.isdir(source):
            return self._load_image_folder(source,labels,classes=classes)
        if isinstance(source,list) and all(isinstance(s,str) for s in source):
            return self._load_text(source,labels)
        if isinstance(source,np.ndarray):
            return self._load_image_array(source,labels)
        raise ValueError(f"Unsupported source type: {type(source)}")

    def preprocess_single(self,sample,modality):
        if modality == 'image_path':
            img = keras_image.load_img(sample,target_size=self.target_image_size)
            arr = keras_image.img_to_array(img).astype('float32') / 255.0
            return tf.convert_to_tensor(arr[np.newaxis])

        if modality == 'image_array':
            arr = np.array(sample,dtype=np.float32)
            if arr.max() > 1.0:
                arr /= 255.0
            if arr.ndim == 2:
                arr = arr[:, :, np.newaxis]
            return tf.convert_to_tensor(arr[np.newaxis])

        if modality == 'text':
            vec = self.vectorizer.transform([sample]).toarray().astype('float32')
            return tf.convert_to_tensor(vec)
        raise ValueError(f"Sorry. Unknown modality: {modality}")

    def _load_csv(self,path,label_col=None):
        df = pd.read_csv(path)
        print(f"CSV has been loaded. Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        if label_col is None:
            label_col = input("Please specify which column the label is and enter the column's name: ").strip()
        if label_col not in df.columns:
            raise ValueError(f"Column '{label_col}' not found. Available: {list(df.columns)}")
        y    = df[label_col].values.astype('int32')
        X    = df.drop(columns=[label_col]).values.astype('float32')
        print(f"Features shape: {X.shape}, Labels shape: {y.shape}")
        meta = {
            'modality' : 'csv',
            'shape' : (X.shape[1],),
            'feature_names': [c for c in df.columns if c != label_col]
        }
        return X, y, meta

    def _load_image_folder(self,root,labels=None,classes=None):
        X, y = [], []
        all_sub = sorted([d for d in os.listdir(root)
                          if os.path.isdir(os.path.join(root,d))])
        if classes is not None:
            sub = [c for c in classes if c in all_sub]
            missing = [c for c in classes if c not in all_sub]
            if missing:
                raise ValueError(f"We're sorry. These class folders were not found in {root}: {missing}")
        else:
            sub = all_sub

        if sub:
            print(f"Currently loading classes: {sub}")
            for class_idx, class_name in enumerate(sub):
                folder = os.path.join(root,class_name)
                imgs, lbs = self._read_images(folder,class_idx)
                print(f"  class {class_idx} '{class_name}': {len(imgs)} images")
                X.extend(imgs); y.extend(lbs)
        else:
            if labels is None:
                raise ValueError("Sorry. A flat folder requires an explicit labels array.")
            imgs, _ = self._read_images(root,label=0)
            X.extend(imgs)
            y = list(labels)

        X = np.array(X, dtype='float32')
        y = np.array(y, dtype='int32')
        meta = {'modality':'image','shape':X.shape[1:],'classes':sub}
        return X, y, meta

    def _read_images(self,folder,label):
        data, labs = [], []
        for fname in os.listdir(folder):
            if not fname.lower().endswith(('.jpg','.jpeg','.png')):
                continue
            path = os.path.join(folder, fname)
            img = keras_image.load_img(path,target_size=self.target_image_size)
            arr = keras_image.img_to_array(img).astype('float32') / 255.0
            data.append(arr)
            labs.append(label)
        return data, labs

    def _load_image_array(self,arr,labels):
        arr = arr.astype('float32')
        if arr.max() > 1.0:
            arr /= 255.0
        y = np.array(labels,dtype='int32') if labels is not None else None
        meta = {'modality':'image','shape': arr.shape[1:]}
        return arr, y, meta

    def _load_text(self,texts,labels):
        X = self.vectorizer.fit_transform(texts).toarray().astype('float32')
        y = np.array(labels, dtype='int32') if labels is not None else None
        meta = {
            'modality' : 'text',
            'vocab_size' : X.shape[1],
            'feature_names': self.vectorizer.get_feature_names_out()
        }
        return X, y, meta

In [48]:
def integrated_gradients(model,sample_input,sample_label,m_steps=50,baseline=None):
    sample = tf.squeeze(tf.cast(sample_input,tf.float32),axis=0)
    baseline = tf.zeros_like(sample_input) if baseline is None else tf.squeeze(tf.cast(baseline,tf.float32),axis=0)

    alphas = tf.linspace(0.0,1.0,m_steps + 1)
    broadcast = tf.reshape(alphas, [-1] + [1] * (len(sample_input.shape)))
    interpolated = baseline + broadcast * (sample_input - baseline)

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        preds = model(interpolated)
        probs = tf.nn.softmax(preds,axis=-1)[:, sample_label]
    grads = tape.gradient(probs,interpolated)

    avg_grads = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(avg_grads, axis=0)
    return (sample - baseline) * avg_grads

In [49]:
# notebook debugging
def visualize_image_ig(sample,attribution,title="IG heatmap"):
    img  = tf.squeeze(sample).numpy()
    attr = tf.squeeze(attribution).numpy()

    heatmap = np.abs(attr).sum(axis=-1) if attr.ndim == 3 else np.abs(attr)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    fig, axes = plt.subplots(1,2,figsize=(8,4))
    axes[0].imshow(img,cmap='gray' if img.ndim == 2 else None)
    axes[0].set_title("Original"); axes[0].axis('off')
    axes[1].imshow(img,cmap='gray' if img.ndim == 2 else None)
    axes[1].imshow(heatmap,cmap='inferno',alpha=0.55)
    axes[1].set_title(title); axes[1].axis('off')
    plt.tight_layout(); plt.show()

def visualize_feature_ig(attribution,feature_names=None,top_k=15,title="Feature attributions"):
    scores = tf.squeeze(attribution).numpy()
    if feature_names is None:
        feature_names = [f"feat_{i}" for i in range(len(scores))]

    idx = np.argsort(np.abs(scores))[::-1][:top_k]
    names = [feature_names[i] for i in idx]
    values = scores[idx]
    colors = ['steelblue' if v > 0 else 'tomato' for v in values]

    plt.figure(figsize=(8,4))
    plt.barh(names[::-1],values[::-1],color=colors[::-1])
    plt.axvline(0,color='black',linewidth=0.8)
    plt.title(title); plt.tight_layout(); plt.show()

In [50]:
class ModelBuilder:
    def build(meta,y,custom_layers=None):
        modality = meta['modality']
        shape = meta['shape']

        if modality == 'image':
            layers = ModelBuilder._image_layers(shape,custom_layers)
        elif modality in ('text', 'csv'):
            layers = ModelBuilder._dense_layers(shape,custom_layers)
        else:
            raise ValueError(f"Unknown modality: {modality}")
        # pass model_output=y so MLP infers num_of_classes itself how it does in MLPClass
        mlp = MLP(model_output=y,layers=layers)
        mlp.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
        return mlp

    def _image_layers(input_shape,custom=None):
        if custom:
            return custom
        return [
            Layer(input_shape=input_shape, layer_type='Input'),
            Conv(dimension=2, filters=16, kernel_size=(3, 3),
                 activation='relu', layer_type='Conv2D'),
            Layer(layer_type='MaxPool2D'),
            Conv(dimension=2, filters=32, kernel_size=(3, 3),
                 activation='relu', layer_type='Conv2D'),
            Layer(layer_type='MaxPool2D'),
            Layer(layer_type='Flatten'),
            Layer(neurons=64, activation='relu', layer_type='Dense'),
        ]

    def _dense_layers(input_shape,custom=None):
        if custom:
            return custom
        feat = input_shape[0]
        return [
            Layer(input_shape=input_shape, neurons=max(feat, 16),
                  activation='relu', layer_type='Dense'),
            Layer(neurons=32, activation='relu', layer_type='Dense'),
        ]

In [51]:
class ExplainabilityWrapper:
    def __init__(self,mlp,dataset,modality,loader):
        self.mlp = mlp        # trained MLP instance
        self.model = mlp.model  # raw keras model
        self.dataset = dataset    # pre-loaded X array
        self.modality = modality   # 'image' or 'text'
        self.loader = loader     # DataLoader instance for preprocess_single

    def fetch_input(self,index):
        return self.dataset[index]

In [52]:
class IGPipeline:
    def __init__(self,model,target_image_size=(64, 64)):
        self.loader = DataLoader(target_image_size=target_image_size)
        self.model = model
        self.meta = None
        self._fitted = False
        self.wrapper = None
        self.dataset = None
        # maybe add model in the input param which will be an instance of MLP from MLPClass

    def fit(self,source,labels=None,epochs=10,custom_layers=None,classes=None,**fit_kwargs):
        X, y, self.meta = self.loader.load(source, labels, classes=classes)
        self.dataset = X
        #self.model = ModelBuilder.build(self.meta, y, custom_layers) # just use MLP model
        #self.model.fit(X, y, epochs=epochs, **fit_kwargs) #
        self._fitted = True
        self.wrapper = ExplainabilityWrapper(
            mlp = self.model,
            dataset = X,
            modality = self.meta['modality'],
            loader = self.loader
        )
        return self

    def explain(self,sample,modality=None,m_steps=50):
        if not self._fitted:
            raise RuntimeError("Call fit() before explain().")
        modality = modality or self._detect_modality(sample)
        tensor = self.loader.preprocess_single(sample,modality)
        pred_idx = int(np.argmax(self.model.predict(tensor), axis=-1)[0])
        print(f"Predicted class: {pred_idx}")
        attr = integrated_gradients(self.model.model, tensor, pred_idx, m_steps)
        if modality in ('image_path','image_array'):
            visualize_image_ig(tensor, attr)
        else:
            names = self.meta.get('feature_names')
            visualize_feature_ig(attr, feature_names=names)
        return attr

    def _detect_modality(self,sample):
        if isinstance(sample,str) and os.path.isfile(sample):
            return 'image_path'
        if isinstance(sample,str):
            return 'text'
        if isinstance(sample,np.ndarray) and sample.ndim >= 3:
            return 'image_array'
        if isinstance(sample,np.ndarray):
            return 'tabular'
        raise ValueError(f"Sorry. We cannot auto-detect modality for type {type(sample)}")

In [53]:
def _get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D,
                               tf.keras.layers.Conv1D,
                               tf.keras.layers.Conv3D)):
            return layer.name
    return None
#def output_integrated_gradients(wrapper,image_index,m_steps=50): ## THE older function definition w/ wrapper
def output_integrated_gradients(model,sample_input,sample_label,m_steps=50,baseline=None):
    # raw = wrapper.fetch_input(image_index)

    # if wrapper.modality == 'text':
    #     sample = wrapper.loader.preprocess_single(raw,'text')
    # else:
    #     arr = np.array(raw,dtype=np.float32)
    #     if arr.max() > 1.0:
    #         arr /= 255.0
    #     if arr.ndim == 2:
    #         arr = arr[:, :,np.newaxis]
    #     sample = tf.convert_to_tensor(arr[np.newaxis])

    # layer_name = _get_last_conv_layer(wrapper.model)
    # if layer_name:
    #     print(f"Last conv layer: {layer_name}")
    # else:
    #     print("No conv layer found - dense-only architecture.")

    pred_idx = int(np.argmax(wrapper.model.predict(sample),axis=-1)[0])
    # print(f"Predicted class: {pred_idx}")

    attr = integrated_gradients(model,sample_input,sample_output,pred_idx,m_steps)
    attr_np = tf.squeeze(attr).numpy()
    heatmap = np.abs(attr_np).sum(axis=-1) if attr_np.ndim == 3 else np.abs(attr_np)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    original = tf.squeeze(sample_input).numpy()
    overlay_display = (original,heatmap)

    return heatmap, overlay_display

In [54]:
#mlp.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#mlp.fit(X_train,y_train,epochs=3)
pipe = IGPipeline(model=mlp,target_image_size=(64,64))
pipe.fit('/Users/graceg/Desktop/',labels=None, epochs=5,classes=['cats','dogs']) # add fish, horses, birds ...
pipe.explain('/Users/graceg/Desktop/cats/cat1.jpg')

heatmap, overlay_display = output_integrated_gradients(pipe.wrapper,image_index=0)

Currently loading classes: ['cats', 'dogs']
  class 0 'cats': 2 images
  class 1 'dogs': 3 images


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_25" is incompatible with the layer: expected axis -1 of input shape to have value 12544, but received input with shape (1, 196608)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 64, 64, 3), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>